In [1]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
import sys
import os
sys.path.append(os.path.abspath("../.."))
from mlforecast.lag_transforms import RollingMean, RollingStd
import lightgbm as lgb
import holidays
from tinyshift.modelling import FirstStageForecasterEvaluator, TwoStageForecasterEvaluator, TwoStageForecasterWrapper

In [2]:
def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):

    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            base_demand = np.random.uniform(0.5, 5.0)
            
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365)

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [3]:
def temporal_split_by_horizon(df, time_col='ds', horizon_days=28):
    df = df.sort_values(time_col)
    max_date = df[time_col].max()
    cutoff_date = max_date - pd.Timedelta(days=horizon_days)
    
    df_train = df[df[time_col] <= cutoff_date].copy()
    df_test = df[df[time_col] > cutoff_date].copy()
    
    return df_train, df_test, cutoff_date

In [4]:
us_holidays = holidays.US(years=range(2022, 2027))

_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1): 
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


def is_holiday_window(dates) -> pd.Series:
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

def add_relative_price(df: pd.DataFrame, id_col: str = 'unique_id', price_col: str = 'sell_price') -> pd.DataFrame:
    df = df.copy()
    
    mean_price_per_sku = df.groupby(id_col)[price_col].transform('mean')
    
    df['relative_price'] = df[price_col] / (mean_price_per_sku + 1e-6)
    
    return df

df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])
df_nixtla = add_relative_price(df_nixtla)
df_train, df_test, cutoff = temporal_split_by_horizon(df_nixtla, horizon_days=28)

In [141]:
fcst = MLForecast(
    models={
        'lgb_issm': lgb.LGBMRegressor(
            objective='poisson',
            metric='poisson',
            n_estimators=100,
            learning_rate=1e-3,
            random_state=42,
            verbosity=-1,
        )
    },
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [ ]:
tsf = TwoStageForecasterWrapper(fcst)
tsf.fit(df_train[["ds", "y", "sell_price", 'relative_price', "is_event", "is_holiday_window", "unique_id"]], h=28, n_windows=10, refit=True)

In [143]:
df_test[["ds", "sell_price", "is_event", "is_holiday_window", "unique_id"]].groupby("unique_id")["ds"].nunique()

unique_id
STORE_01_FOODS_1_001    28
STORE_01_FOODS_1_002    28
STORE_01_FOODS_1_003    28
STORE_01_FOODS_1_004    28
STORE_01_FOODS_1_005    28
STORE_02_FOODS_1_001    28
STORE_02_FOODS_1_002    28
STORE_02_FOODS_1_003    28
STORE_02_FOODS_1_004    28
STORE_02_FOODS_1_005    28
STORE_03_FOODS_1_001    28
STORE_03_FOODS_1_002    28
STORE_03_FOODS_1_003    28
STORE_03_FOODS_1_004    28
STORE_03_FOODS_1_005    28
Name: ds, dtype: int64

In [144]:
df_res = tsf.pmf(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], max_k=4)
df_res.loc[:, "y"] = df_test["y"].values

In [145]:
df_res

,unique_id,ds,lambda_t,r_dispersion,P(Y=0),P(Y=1),P(Y=2),P(Y=3),P(Y=4),P(Y>4),y
0,STORE_01_FOODS_1_001,2023-12-04,2.495951,10.910460,0.105644,0.214592,0.237923,0.190625,0.123420,0.127796,1
1,STORE_01_FOODS_1_001,2023-12-05,2.495951,10.910460,0.105644,0.214592,0.237923,0.190625,0.123420,0.127796,5
2,STORE_01_FOODS_1_001,2023-12-06,2.568284,10.910460,0.099621,0.207103,0.235006,0.192705,0.127693,0.137871,1
3,STORE_01_FOODS_1_001,2023-12-07,2.565290,10.910460,0.099863,0.207410,0.235132,0.192626,0.127521,0.137449,3
4,STORE_01_FOODS_1_001,2023-12-08,2.668153,10.910460,0.091912,0.197047,0.230581,0.194984,0.133241,0.152235,3
...,...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,2.510106,3.486235,0.150969,0.220318,0.206875,0.158368,0.107499,0.155970,4
416,STORE_03_FOODS_1_005,2023-12-28,2.510106,3.486235,0.150969,0.220318,0.206875,0.158368,0.107499,0.155970,0
417,STORE_03_FOODS_1_005,2023-12-29,2.623533,3.486235,0.141422,0.211707,0.203915,0.160126,0.111496,0.171335,2
418,STORE_03_FOODS_1_005,2023-12-30,2.623533,3.486235,0.141422,0.211707,0.203915,0.160126,0.111496,0.171335,2


In [146]:
df_res = tsf.marginal_cost(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], underage_cost=200, overage_cost=100, max_k=4)
df_res.loc[:, "y"] = df_test["y"].values
df_res

,unique_id,ds,lambda_t,r_dispersion,MC(k=0),MC(k=1),MC(k=2),MC(k=3),MC(k=4),y
0,STORE_01_FOODS_1_001,2023-12-04,2.495951,10.910460,200.0,168.306657,103.929070,32.552311,-24.635158,1
1,STORE_01_FOODS_1_001,2023-12-05,2.495951,10.910460,200.0,168.306657,103.929070,32.552311,-24.635158,5
2,STORE_01_FOODS_1_001,2023-12-06,2.568284,10.910460,200.0,170.113745,107.982762,37.480928,-20.330612,1
3,STORE_01_FOODS_1_001,2023-12-07,2.565290,10.910460,200.0,170.041215,107.818238,37.278658,-20.509237,3
4,STORE_01_FOODS_1_001,2023-12-08,2.668153,10.910460,200.0,172.426446,113.312336,44.137994,-14.357289,3
...,...,...,...,...,...,...,...,...,...,...
415,STORE_03_FOODS_1_005,2023-12-27,2.510106,3.486235,200.0,154.709251,88.613752,26.551197,-20.959187,4
416,STORE_03_FOODS_1_005,2023-12-28,2.510106,3.486235,200.0,154.709251,88.613752,26.551197,-20.959187,0
417,STORE_03_FOODS_1_005,2023-12-29,2.623533,3.486235,200.0,157.573498,94.061518,32.887156,-15.150784,2
418,STORE_03_FOODS_1_005,2023-12-30,2.623533,3.486235,200.0,157.573498,94.061518,32.887156,-15.150784,2


In [147]:
df_res = tsf.predict(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], quantiles=[0.05, 0.50, 0.95, 0.99])
df_res.loc[:, "y"] = df_test["y"].values

In [148]:
FirstStageForecasterEvaluator.evaluate(df_res)

,Metrics
PBias,3.9400
False Demand on Zero-Days (Avg Pred),2.5561
Peak Demand Deviation (%),-16.8700


In [149]:
TwoStageForecasterEvaluator.evaluate(df_res, quantiles=[0.05, 0.50, 0.95, 0.99])

,Pinball Loss,Target Coverage,Empirical Coverage,Coverage Gap
q_5,0.1229,0.05,0.2000,0.1500
q_50,0.8833,0.50,0.5738,0.0738
q_95,0.3310,0.95,0.9452,-0.0048
q_99,0.1111,0.99,0.9857,-0.0043


In [150]:
import joblib

model_path = "tsf.joblib"

# Save the fitted wrapper (works the same for mode="local" or mode="global").
joblib.dump(tsf, model_path)

# Load it back into a new object.
loaded_forecast = joblib.load(model_path)

In [151]:
loaded_forecast.predict(h=12, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]])

,unique_id,ds,lambda_t,r_dispersion,q_5,q_50,q_95
0,STORE_01_FOODS_1_001,2023-12-04,2.495951,10.910460,0,2,6
1,STORE_01_FOODS_1_001,2023-12-05,2.495951,10.910460,0,2,6
2,STORE_01_FOODS_1_001,2023-12-06,2.568284,10.910460,0,2,6
3,STORE_01_FOODS_1_001,2023-12-07,2.565290,10.910460,0,2,6
4,STORE_01_FOODS_1_001,2023-12-08,2.668153,10.910460,0,2,6
...,...,...,...,...,...,...,...
175,STORE_03_FOODS_1_005,2023-12-11,2.568284,3.486235,0,2,7
176,STORE_03_FOODS_1_005,2023-12-12,2.568284,3.486235,0,2,7
177,STORE_03_FOODS_1_005,2023-12-13,2.552395,3.486235,0,2,7
178,STORE_03_FOODS_1_005,2023-12-14,2.565290,3.486235,0,2,7
